In [1]:
"""Safe artifact exporters for pipeline results and notifications."""
from __future__ import annotations
from pathlib import Path
import json
from typing import Any

def _safe(value:Any)->Any:
    if isinstance(value,Path): return str(value)
    if isinstance(value,dict): return {str(k):_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)): return [_safe(v) for v in value]
    try: json.dumps(value); return value
    except TypeError: return str(value)

def export_results(results:dict, output_dir:str|Path)->dict:
    """Write analysis_results.json, execution_log.json, and email_summary.txt."""
    out=Path(output_dir); out.mkdir(parents=True,exist_ok=True)
    analysis=out/"analysis_results.json"; log=out/"execution_log.json"; email=out/"email_summary.txt"
    analysis.write_text(json.dumps(_safe(results),indent=2),encoding="utf-8")
    log.write_text(json.dumps(_safe(results.get("execution_log",[])),indent=2),encoding="utf-8")
    k=results.get("kpis",{}); body=(f"Subject: Business Analysis Report\nRecipient: recipient@example.com\n\n"
        f"Status: {results.get('status','unknown')}\nRevenue: {k.get('total_revenue','unavailable')}\nOrders: {k.get('total_orders','unavailable')}\n"
        f"Report attachment: {results.get('pdf_path','unavailable')}\n")
    email.write_text(body,encoding="utf-8")
    return {"analysis_results":str(analysis),"execution_log":str(log),"email_summary":str(email)}
